# Decision Tree: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains a single decision tree to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It uses the same data, inputs and time-based folds as the random forest notebook, so the two can be compared directly. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions
3. trains a first decision tree and checks it on the calibration split
4. tunes the tree's settings with a randomized search over time-based folds
5. saves the best model

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`, `SimpleImputer`), the model (`DecisionTreeClassifier`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **time / joblib:** timing the search and saving the final model to a file

In [23]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [24]:
train = pd.read_parquet("../../data/modeling/modeling_train.parquet")
calib = pd.read_parquet("../../data/modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (169827, 93)
Calibration shape: (297009, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,1050A093F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30
1,1050A0ANF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
2,1080A06JF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,6.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
3,1080A075F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,7.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,681.0,30
4,1080A09LF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [25]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `dt_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** fills blanks with the **training** median and adds a "was missing" column for each column that had blanks. The blanks come from specific drive models, such as S197 on two Toshiba models, so the fact that a value is missing is itself useful.
   - **vendor:** one-hot encoded

Because this sits inside a pipeline, the medians are learned from training data only. They're then re-used on the calibration split, so no information leaks from calibration into training.

This is the same preprocessing as the random forest. A tree splits on one column at a time against a cutoff, so the columns don't need scaling.

In [26]:
dt_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First Decision Tree (Before Tuning)
Builds the full model, preprocessing followed by a single decision tree, with reasonable starting settings:

- **`max_depth=10`:** the tree can ask at most 10 questions in a row before deciding. A single tree with no limit keeps splitting until it has memorized the training data, and unlike a forest there are no other trees to average that out.
- **`min_samples_leaf=20`:** each leaf must contain at least 20 windows, so no decision rests on a handful of drives
- **`class_weight="balanced"`:** gives failures more weight. (A forest's `"balanced_subsample"` option doesn't exist for a single tree, because there are no per-tree samples.)
- **`random_state=42`:** makes results repeatable

Every split looks at all the columns, which is the default for a single tree. The random column subsets a forest uses are only there to make its trees different from each other.

In [27]:
dt_model = Pipeline([
    ("preprocessing", dt_preprocess),
    ("classifier", DecisionTreeClassifier(
        max_depth = 10,
        min_samples_leaf = 20,
        class_weight = 'balanced',
        random_state = 42 
    ))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing learns its medians, then the tree is grown.

In [28]:
dt_model.fit(X_train_raw, y_train)

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('features',
                                  FunctionTransformer(func=<function build_features at 0x124d32c00>)),
                                 ('columns',
                                  ColumnTransformer(transformers=[('numeric',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'),
                                                                   ['s1_first',
                                                                    's1_last',
                                                                    's1_delta',
                                                                    's3_first',
                                                                    's3_last',
                                                                    's3_delta',
                                                                    's4_first',
                                                                    's4_last',
                                                                    's4_delta',
                                                                    's5_first',
                                                                    's5_last',...
                                                                    's194_last',
                                                                    's194_delta',
                                                                    's197_first',
                                                                    's197_last',
                                                                    's197_delta', ...]),
                                                                  ('vendor',
                                                                   OneHotEncoder(categories=[['Seagate',
                                                                                              'Toshiba',
                                                                                              'WDC',
                                                                                              'HGST',
                                                                                              'Other']],
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False),
                                                                   ['vendor'])],
                                                    verbose_feature_names_out=False))])),
                ('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_leaf=20,
                                        random_state=42))])

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

A single tree's "probability" is just the share of failures in the leaf a window lands in, so it only takes as many distinct values as the tree has leaves. Expect precision and recall to jump in steps between thresholds rather than change smoothly as they do for the forest.

**Result:**
- **Ranking:** PR-AUC is **0.0701**, against 0.0073 for the SMART 5 rule, 0.0011 for random, 0.082 for logistic regression and 0.0995 for the untuned random forest. So one tree ranks failing drives 10 times better than the SMART 5 rule, but below both logistic regression and the forest. ROC-AUC is 0.879.
- **Thresholds:** the best F1 (0.152) is at 1.0, where precision is 16.7% and recall 14.0%. At 1.0 the tree only flags windows that land in leaves holding nothing but failures. At the default 0.5, precision is only 1.3% at 74.4% recall. As with the forest, the probabilities are too high because training has 43× more failures than reality, so a threshold has to be chosen deliberately and the probabilities recalibrated later.

In [29]:
results=[]
y_prob = dt_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "Decision Tree",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,Decision Tree,0.0701,0.8791,0.0011,1.0000,0.0022,0.092
1,0.1,Decision Tree,0.0701,0.8791,0.0011,0.9909,0.0022,0.092
2,0.2,Decision Tree,0.0701,0.8791,0.0017,0.9299,0.0034,0.092
3,0.3,Decision Tree,0.0701,0.8791,0.0027,0.8476,0.0054,0.092
4,0.4,Decision Tree,0.0701,0.8791,0.0120,0.7591,0.0237,0.092
5,0.5,Decision Tree,0.0701,0.8791,0.0131,0.7439,0.0258,0.092
6,0.6,Decision Tree,0.0701,0.8791,0.0165,0.7287,0.0323,0.092
7,0.7,Decision Tree,0.0701,0.8791,0.0188,0.7256,0.0367,0.092
8,0.8,Decision Tree,0.0701,0.8791,0.0299,0.6890,0.0573,0.092
9,0.9,Decision Tree,0.0701,0.8791,0.0373,0.6524,0.0706,0.092


## 6. Tuning the Decision Tree
### 6.1 Search Setup and Time-Based Folds
Sets up the tuning:

- **`dt_model`:** the search reuses the model from section 5. It works on a fresh copy, so the first model is left untouched, and each combination overwrites the settings being searched. `class_weight="balanced"` and `random_state=42` carry over to every combination.
- **`param_distributions`:** the settings to try:
  - split rule (`criterion`): Gini impurity or entropy
  - maximum tree depth
  - minimum windows needed to split a node, and to form a leaf
  - share of columns tried at each split
  - maximum number of leaves
  - pruning strength (`ccp_alpha`): after the tree is grown, cuts back branches whose extra accuracy isn't worth their added complexity. 0 means no pruning; larger values give smaller trees.

  Together that's 4,320 combinations.
- **`time_folds()`:** builds 4 cross-validation folds **by time**. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead. This mirrors real use: predicting the future from the past.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [30]:
param_distributions = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__ccp_alpha": [0, 1e-5, 1e-3, 1e-4],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 10, 20],
    "classifier__min_samples_leaf": [50, 100, 200, 500],
    "classifier__max_features": ["sqrt", 0.5, None],
    "classifier__max_leaf_nodes": [None, 100, 300]
}

## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **100 random combinations** from the 4,320. Each is trained once per fold, giving **400 fits**, and scored by PR-AUC (`average_precision`). A single tree trains in seconds and uses little memory, so unlike the forest the search runs several fits at once (`n_jobs=-1`) and can afford to try more combinations.
- **Refit:** the best combination is retrained on all of the training data (`refit=True`).
- **Results table:**
  - `mean_test_score`: the average PR-AUC over the 4 folds
  - `std_test_score`: how much the 4 fold scores vary
  - `rank_test_score`: 1 = best

  The table is saved to `dt_search_results_ranked.csv`.
- **Save:** the best model is saved to `dt_best.joblib`, so it can be reloaded without retraining.

**Result:**
- **Time:** the search took **0.5 minutes**, against 36 for the forest.
- **Best settings:** **depth 15, `min_samples_leaf=100`, `min_samples_split=10`, entropy, all features, no leaf limit, `ccp_alpha=1e-5`**, with a mean time-CV PR-AUC of **0.548** (std 0.033), against 0.573 for the tuned forest.
- **Leaf size:** a first search with leaf sizes of 10 to 100 picked 100, the edge of the range, so the range was widened to 50 to 500 and the search re-run. Leaves of 200 and 500 scored worse, so 100 is a real best and not a limit of the search.
- **Close calls:** the top 5 settings are within 0.008 of each other, well under one std apart, so they perform about the same. Pruning barely matters: the best tree uses almost none, and `ccp_alpha=1e-3` never reaches the top 10.
- **A readable alternative:** the best depth-5 tree scores 0.530, about half a std below the best. It is small enough to draw in full, which is the main reason to use a single tree at all.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other.

In [31]:
search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_distributions,
    n_iter=100,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,
    n_jobs=-1,
    refit="pr_auc",
    random_state=42,
    verbose=2,
    return_train_score=False,
)

start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))

results.to_csv("dt_search_results_ranked.csv", index=False)

# Save the best model (chosen by PR-AUC, refit on all of train)
best_dt = search.best_estimator_
joblib.dump(best_dt, "dt_best.joblib")


Fitting 4 folds for each of 100 candidates, totalling 400 fits
[CV] END classifier__ccp_alpha=0, classifier__criterion=gini, classifier__max_depth=None, classifier__max_features=sqrt, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=100, classifier__min_samples_split=20; total time=   0.2s
[CV] END classifier__ccp_alpha=0, classifier__criterion=gini, classifier__max_depth=10, classifier__max_features=sqrt, classifier__max_leaf_nodes=None, classifier__min_samples_leaf=50, classifier__min_samples_split=10; total time=   0.2s
[CV] END classifier__ccp_alpha=0, classifier__criterion=gini, classifier__max_depth=None, classifier__max_features=sqrt, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=100, classifier__min_samples_split=20; total time=   0.3s
[CV] END classifier__ccp_alpha=0.001, classifier__criterion=gini, classifier__max_depth=10, classifier__max_features=sqrt, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=100, classifier__min_samples_split=

/opt/homebrew/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END classifier__ccp_alpha=1e-05, classifier__criterion=gini, classifier__max_depth=5, classifier__max_features=None, classifier__max_leaf_nodes=None, classifier__min_samples_leaf=500, classifier__min_samples_split=2; total time=   0.9s
[CV] END classifier__ccp_alpha=0.001, classifier__criterion=gini, classifier__max_depth=10, classifier__max_features=sqrt, classifier__max_leaf_nodes=None, classifier__min_samples_leaf=500, classifier__min_samples_split=20; total time=   0.2s
[CV] END classifier__ccp_alpha=0.001, classifier__criterion=gini, classifier__max_depth=5, classifier__max_features=None, classifier__max_leaf_nodes=None, classifier__min_samples_leaf=200, classifier__min_samples_split=2; total time=   1.1s
[CV] END classifier__ccp_alpha=0.0001, classifier__criterion=gini, classifier__max_depth=None, classifier__max_features=sqrt, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=200, classifier__min_samples_split=10; total time=   0.5s
[CV] END classifier__ccp_alpha

,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_leaf_nodes,param_classifier__max_features,param_classifier__max_depth,param_classifier__criterion,param_classifier__ccp_alpha,PR_AUC_rank
0,0.547704,0.032707,0.229254,0.050208,0.715611,0.017781,0.344678,0.061494,1.304936,10,100,None,None,15,entropy,0.00001,1
1,0.544722,0.037901,0.284741,0.067452,0.702557,0.025135,0.401546,0.073840,0.594958,2,50,None,0.5,10,gini,0.00000,2
2,0.542780,0.030731,0.192402,0.043486,0.727043,0.013269,0.301736,0.055371,1.487749,20,50,None,None,20,entropy,0.00001,3
3,0.541902,0.029017,0.250250,0.068923,0.704260,0.022117,0.364473,0.078472,0.636526,20,100,300,0.5,10,entropy,0.00001,4
4,0.540268,0.021662,0.223359,0.057056,0.716925,0.015268,0.337086,0.069270,1.257764,2,100,300,None,15,gini,0.00000,5
5,0.536368,0.035837,0.194123,0.045350,0.729923,0.015622,0.304263,0.057585,1.195191,10,200,None,None,20,entropy,0.00001,6
6,0.532955,0.021243,0.205391,0.036287,0.724191,0.016864,0.318413,0.044604,0.364629,20,100,300,sqrt,20,entropy,0.00000,7
7,0.532618,0.026658,0.179597,0.050079,0.748762,0.021469,0.286146,0.065443,1.456095,10,50,300,None,None,entropy,0.00010,8
8,0.532220,0.034212,0.293361,0.044296,0.687440,0.035914,0.410296,0.049600,0.331138,10,50,None,sqrt,10,entropy,0.00001,9
9,0.531444,0.028520,0.279935,0.063820,0.699701,0.026956,0.396331,0.068448,0.322116,20,50,300,sqrt,10,gini,0.00000,10


['dt_best.joblib']